#Lab 2: Group and create

In [7]:
import pandas as pd
import numpy as np
listings = pd.read_csv(r'C:\Users\Hey HP\Desktop\gowthami_py\Data Visualisation and Analysis\listings.csv')

In [8]:
#Task 1: Count property types
# Five most common property types
listings['property_type'].value_counts().head(5)

# Share of Entire rental unit
listings['property_type'].value_counts(normalize=True).round(3).head(1)

property_type
Entire rental unit    0.534
Name: proportion, dtype: float64

In [9]:
#Task 2: Smallest neighbourhoods
listings['neighbourhood_cleansed'].value_counts().tail(3)

neighbourhood_cleansed
Rinkeby-Tensta    108
Skärholmens        96
Spånga-Tensta      89
Name: count, dtype: int64

In [10]:
#Task 3: Average beds per room type
listings.groupby('room_type')['beds'].mean().round(2)
listings.groupby('room_type')['beds'].agg(['count', 'mean']).round(2)

,count,mean
room_type,,
Entire home/apt,3680,2.48
Hotel room,3,1.00
Private room,766,1.48
Shared room,24,4.58


In [11]:
#Task 4: Reviews per neighbourhood
(listings.groupby('neighbourhood_cleansed')['number_of_reviews']
         .agg(['count', 'mean', 'median'])
         .round(2)
         .sort_values('median'))

,count,mean,median
neighbourhood_cleansed,,,
Skärholmens,96,21.29,3.0
Farsta,177,18.18,4.0
Hässelby-Vällingby,148,14.94,4.0
Rinkeby-Tensta,108,16.44,5.0
Bromma,281,19.16,5.0
Enskede-Årsta-Vantörs,445,22.55,7.0
Spånga-Tensta,89,17.01,7.0
Hägersten-Liljeholmens,464,22.92,8.0
Skarpnäcks,221,22.55,8.0


In [12]:
#Task 5: Own summary table
summary = listings.groupby('room_type').agg(
    n_listings=('id', 'count'),
    avg_beds=('beds', 'mean'),
    max_reviews=('number_of_reviews', 'max')
).round(2)
summary

,n_listings,avg_beds,max_reviews
room_type,,,
Entire home/apt,3760,2.48,1293
Hotel room,3,1.00,79
Private room,787,1.48,867
Shared room,24,4.58,510


In [13]:
#Task 6: Crosstab
ct = pd.crosstab(listings['neighbourhood_cleansed'],
                 listings['host_is_superhost'],
                 normalize='index')
(ct * 100).round(1)

(ct * 100).round(1)['t'].sort_values()

neighbourhood_cleansed
Bromma                     7.1
Farsta                     9.0
Kungsholmens               9.8
Enskede-Årsta-Vantörs     11.7
Hässelby-Vällingby        12.2
Hägersten-Liljeholmens    12.5
Rinkeby-Tensta            13.0
Skarpnäcks                13.1
Skärholmens               15.6
Östermalms                16.8
Norrmalms                 19.0
Älvsjö                    19.6
Spånga-Tensta             22.5
Södermalms                23.8
Name: t, dtype: float64

In [14]:
#Task 7: New columns
# (a)
listings['has_reviews'] = listings['number_of_reviews'] > 0
(~listings['has_reviews']).sum()          # 644 are False

# (b)
listings['popularity'] = np.where(listings['number_of_reviews'] >= 50, 'popular', 'other')
listings['popularity'].value_counts()      # other 3805, popular 769

popularity
other      3805
popular     769
Name: count, dtype: int64

In [ ]:
#Task 8: Turn numbers into groups
# On purpose: wrong bins
wrong = pd.cut(listings['minimum_nights'], bins=[1, 6, 29, 500])
wrong.isna().sum()     

np.int64(1331)

In [17]:
listings['stay'] = pd.cut(listings['minimum_nights'],
                          bins=[0, 1, 6, 29, 500],
                          labels=['1 night', '2-6 nights', '7-29 nights', '30+ nights'])

listings['stay'].value_counts().sort_index()
listings['stay'].isna().sum()      

np.int64(4)

In [ ]:
#Part 2: Go deeper

#Task 1: Who has never been reviewed?

listings['has_reviews'] = listings['number_of_reviews'] > 0
listings['no_reviews'] = ~listings['has_reviews']

listings['no_reviews'].mean()    

share = (listings.groupby('neighbourhood_cleansed')
                 .agg(n=('id', 'count'), share_no_reviews=('no_reviews', 'mean'))
                 .round(3)
                 .sort_values('share_no_reviews'))
share

,n,share_no_reviews
neighbourhood_cleansed,,
Södermalms,1145,0.076
Kungsholmens,420,0.088
Skarpnäcks,221,0.113
Hägersten-Liljeholmens,464,0.134
Älvsjö,214,0.140
Östermalms,297,0.141
Norrmalms,469,0.154
Spånga-Tensta,89,0.191
Enskede-Årsta-Vantörs,445,0.193


In [19]:
#Task 2: Quartiles of reviews
reviewed = listings[listings['number_of_reviews'] >= 1].copy()

reviewed['review_group'] = pd.qcut(reviewed['number_of_reviews'], 4,
                                   labels=['few', 'some', 'many', 'most'])
reviewed['review_group'].value_counts().sort_index()

pd.qcut(reviewed['number_of_reviews'], 4).cat.categories

IntervalIndex([(0.999, 4.0], (4.0, 13.0], (13.0, 36.0], (36.0, 1293.0]], dtype='interval[float64, right]')

In [20]:
#Task 3: Do more reviews mean a better rating?
reviewed.groupby('review_group', observed=True)['review_scores_rating'].mean().round(2)

review_group
few     4.79
some    4.84
many    4.83
most    4.80
Name: review_scores_rating, dtype: float64

In [21]:
#Task 4: Biggest groups, but only if real
combo = (listings.groupby(['neighbourhood_cleansed', 'room_type'])
                 .agg(n=('id', 'count'), avg_accommodates=('accommodates', 'mean'))
                 .round(2))

combo.sort_values('avg_accommodates', ascending=False).head(3)

real = combo[combo['n'] >= 30]
real.sort_values('avg_accommodates', ascending=False).head(3)

,,n,avg_accommodates
neighbourhood_cleansed,room_type,,
Spånga-Tensta,Entire home/apt,54,5.33
Hässelby-Vällingby,Entire home/apt,118,5.25
Farsta,Entire home/apt,145,4.97


In [22]:
#Task 5: Your own question (example)
# Question: Are superhosts' listings more often popular (50+ reviews)
# than other hosts' listings?
listings['popularity'] = np.where(listings['number_of_reviews'] >= 50, 'popular', 'other')
listings['is_popular'] = listings['popularity'] == 'popular'

listings.groupby('host_is_superhost').agg(
    n=('id', 'count'),
    share_popular=('is_popular', 'mean')
).round(3)

# This does NOT tell me whether being a superhost causes more reviews.
# Superhosts may simply have been on the platform longer, and I have not
# checked price, location or listing age.

,n,share_popular
host_is_superhost,,
f,3837,0.101
t,737,0.520
